In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, Markdown

_nb = globals().get("__vsc_ipynb_file__")
ROOT = None
for start in ([Path(_nb)] if _nb else []) + [Path.cwd()]:
    p = start.resolve()
    if p.suffix == ".ipynb":
        p = p.parent
    for candidate in (p, *p.parents):
        if (candidate / "vol_surface.py").is_file():
            ROOT = candidate
            break
    if ROOT:
        break
if not ROOT:
    raise FileNotFoundError("Open the Signal_Monitor folder so vol_surface.py is on the path.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

from app import QuantEngine

INDEX = "SPX"  # only SPX is wired in app.py

engine = QuantEngine()
data = engine.process_index(INDEX)
if not data.get("exists"):
    raise RuntimeError(data.get("error", "pipeline failed"))

print(INDEX, data["date"], "spot", round(data["spot"], 2))
if data.get("warnings"):
    print("warnings:", " | ".join(data["warnings"]))

[pipeline] SPX start
2026-08-26 22:09:59,077 | 31280 | 8764 | [open_context_base.py:411] _init_connect_sync: New connect ready: conn=7498381232759876004(1) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x00000202398638C0>
[chain] US..SPX asof=2026-08-26 spot=7685.01 expiries=30 windows=2
[chain] 2026-08-26..2026-09-24 -> 10426 contracts
[chain] 2026-09-25..2026-10-16 -> 4468 contracts
2026-08-26 22:12:21,285 | 31280 | 14564 | [open_context_base.py:521] on_disconnect: Disconnected: conn=0(1) reason=CallClose
2026-08-26 22:12:26,136 | 31280 | 8764 | [open_context_base.py:411] _init_connect_sync: New connect ready: conn=7498381849570223664(2) context=<futu.quote.open_quote_context.OpenQuoteContext object at 0x000002023A27E360>
2026-08-26 22:12:36,595 | 31280 | 14564 | [open_context_base.py:521] on_disconnect: Disconnected: conn=0(2) reason=CallClose
[pipeline] SPX GEX 8932 contracts, net +10.57B
[pipeline] SPX OK
SPX 2026-08-26 spot 7685.99


In [2]:
GEX_BUCKET = "0"  # None = default (0 / today). Else "0","1","2","3","4","5"


gex = data.get("gex") or {}
bucket_key = GEX_BUCKET or gex.get("default_bucket") or "0"
gex_b = (gex.get("buckets") or {}).get(bucket_key) or {}

rows = [
    ("HMM", "BUY / LONG" if data.get("hmm_signal") else "NEUTRAL / CASH"),
    ("P(low vol today / tmr)", f"{data.get('hmm_prob_today')}% / {data.get('hmm_prob_tmr')}%"),
    ("VRP", f"{data.get('vrp'):.1f} pts"),
    ("Term slope", f"{data.get('tsl'):.1f} vol pts"),
    ("ATM IV 30d", f"{data.get('aiv'):.1f}"),
    ("25d skew", f"{data.get('psk'):.1f}"),
    ("GEX bucket", gex_b.get("label", bucket_key)),
    ("Net GEX", f"{gex_b.get('net_label', '--')} {gex_b.get('regime', '')}"),
    ("Gamma flip", gex_b.get("flip")),
    ("Call wall", gex_b.get("call_wall")),
    ("Put wall", gex_b.get("put_wall")),
]
display(pd.DataFrame(rows, columns=["metric", "value"]))

moves = data.get("hmm_move_table") or []
if moves:
    display(Markdown("**1-sigma move table**"))
    display(pd.DataFrame(moves))

,metric,value
0,HMM,BUY / LONG
1,P(low vol today / tmr),100.0% / 99.8%
2,VRP,2.8 pts
3,Term slope,-2.6 vol pts
4,ATM IV 30d,12.7
5,25d skew,2.5
6,GEX bucket,0 (today)
7,Net GEX,+10.57B long_gamma
8,Gamma flip,7694.83595
9,Call wall,7680.0


**1-sigma move table**

,horizon,implied,historical,spot_implied
0,1d,75(1.0%),46(0.6%),"7,611 | 7,760"
1,15d,290(3.8%),137(1.8%),"7,396 | 7,975"
2,22d,351(4.6%),287(3.7%),"7,335 | 8,036"
3,30d,409(5.3%),326(4.2%),"7,276 | 8,095"


In [9]:
def _misprice_markers(data, x, y, z):
    rows = [
        a for a in (data.get("anomalies") or [])
        if a.get("ks") is not None and a.get("dte") is not None and a.get("surface") == "iv"
    ]
    if not rows:
        return None
    xs, ys, zs, text = [], [], [], []
    for a in rows:
        di = int(np.argmin(np.abs(np.asarray(y) - a["dte"])))
        ki = int(np.argmin(np.abs(np.asarray(x) - a["ks"])))
        xs.append(np.log(a["ks"]))
        ys.append(a["dte"])
        zs.append(z[di][ki] + 0.5)
        text.append(f"{a.get('kind')}: {a.get('detail', '')}")
    return go.Scatter3d(
        x=xs, y=ys, z=zs, mode="markers", name="Mispriced",
        marker=dict(size=5, color="#ff4d4f", symbol="diamond"),
        text=text, hovertemplate="%{text}<br>Log(K/S): %{x:.2f}<br>DTE: %{y:.0f}<extra></extra>",
    )


def _draw_vol_surface(x, y, z, title, colorscale, markers=None):
    fig = go.Figure()
    if len(y) == 1:
        fig.add_trace(go.Scatter3d(
            x=np.log(x), y=[y[0]] * len(x), z=z[0], mode="lines+markers",
            line=dict(color="#2a6cff", width=6),
            marker=dict(color=z[0], colorscale=colorscale, size=4, colorbar=dict(title=title)),
            hovertemplate="Log(K/S): %{x:.2f}<br>DTE: %{y:.0f}d<br>Vol: %{z:.1f}%<extra></extra>",
        ))
    else:
        fig.add_trace(go.Surface(
            x=np.log(x), y=y, z=z, colorscale=colorscale, colorbar=dict(title=title),
            hovertemplate="Log(K/S): %{x:.2f}<br>DTE: %{y:.0f}d<br>Vol: %{z:.1f}%<extra></extra>",
            contours=dict(z=dict(show=True, usecolormap=True, highlightcolor="lime", project=dict(z=True))),
        ))
    if markers is not None:
        fig.add_trace(markers)
    fig.update_layout(
        title=f"Vol Surface — {INDEX} ({title})",
        template="plotly_dark",
        height=560, margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(
            xaxis_title="Log Moneyness ln(K/S)", yaxis_title="DTE", zaxis_title=title,
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8)),
            aspectmode="manual", aspectratio=dict(x=1.0, y=1.2, z=0.6),
        ),
    )
    fig.show()


def plot_surfaces(data):
    x, y = data["surface_x"], data["surface_y"]
    raw = data["surface_z"]
    smooth = data.get("surface_sv") or raw
    _draw_vol_surface(x, y, raw, "Raw Implied Vol (%)", "Viridis", _misprice_markers(data, x, y, raw))
    _draw_vol_surface(x, y, smooth, "SVI Smooth IV (%)", "Cividis", None)
    if data.get("local_vol_available"):
        _draw_vol_surface(x, y, data["surface_w"], "Local Vol (%)", "Magma", None)
    else:
        print("Local vol needs at least two live expiries.")


plot_surfaces(data)

In [4]:
def plot_gex(data, bucket=None):
    gex = data.get("gex") or {}
    buckets = gex.get("buckets") or {}
    key = bucket or gex.get("default_bucket") or "0"
    b = buckets.get(key) or {}
    if not b.get("n_contracts"):
        key = gex.get("default_bucket")
        b = buckets.get(key) or {}
    strikes = np.asarray(b.get("strikes") or [], dtype=float)
    if strikes.size == 0:
        print(f"No GEX in bucket {key!r}.")
        return
    call = np.asarray(b["call_gex"], dtype=float) / 1e9
    put = np.asarray(b["put_gex"], dtype=float) / 1e9
    if b.get("cum_gex") and len(b["cum_gex"]) == len(strikes):
        cum = np.asarray(b["cum_gex"], dtype=float) / 1e9
    else:
        cum = np.cumsum(np.asarray(b.get("net_gex") or (call + put), dtype=float) / 1e9)

    spot = float(data["spot"])
    pad = max(spot * 0.035, 150)
    x0, x1 = spot - pad, spot + pad
    for v in (b.get("call_wall"), b.get("put_wall"), b.get("flip")):
        if v is None or not np.isfinite(v):
            continue
        x0, x1 = min(x0, v - 40), max(x1, v + 40)

    vis = (strikes >= x0) & (strikes <= x1)
    mag = max(0.01, float(np.max(np.abs(np.concatenate([call[vis], put[vis]])))) if vis.any() else 0.01)
    cum_mag = max(0.01, float(np.max(np.abs(cum[vis]))) if vis.any() else 0.01)

    fig = go.Figure()
    fig.add_bar(x=strikes, y=put, name="Put GEX", marker_color="#e74c3c",
                hovertemplate="K %{x:.0f}<br>Put %{y:.2f}B<extra></extra>")
    fig.add_bar(x=strikes, y=call, name="Call GEX", marker_color="#27ae60",
                hovertemplate="K %{x:.0f}<br>Call %{y:.2f}B<extra></extra>")
    fig.add_scatter(x=strikes, y=cum, name="Agg GEX", yaxis="y2",
                    line=dict(color="#5dade2", width=2.5),
                    hovertemplate="K %{x:.0f}<br>Agg %{y:.2f}B<extra></extra>")

    shapes, annotations = [], []
    shapes.append(dict(type="line", xref="x", yref="paper", x0=spot, x1=spot, y0=0, y1=1,
                       line=dict(color="#e8eaed", width=1.5, dash="dash")))
    annotations.append(dict(x=spot, y=1, yref="paper", text=f"Spot {spot:.0f}",
                            showarrow=False, font=dict(color="#e8eaed", size=11),
                            yshift=-14, xanchor="left"))
    if b.get("flip") is not None:
        shapes.append(dict(type="line", xref="x", yref="paper", x0=b["flip"], x1=b["flip"], y0=0, y1=1,
                           line=dict(color="#e67e22", width=1.5)))
        annotations.append(dict(x=b["flip"], y=1, yref="paper", text="Flip",
                                showarrow=False, font=dict(color="#e67e22", size=11), yshift=-8))
    if b.get("call_wall") is not None:
        annotations.append(dict(x=b["call_wall"], y=0, yref="paper",
                                text=f"Call wall {b['call_wall']:.0f}", showarrow=False,
                                font=dict(color="#27ae60", size=11), yshift=12, xanchor="left"))
    if b.get("put_wall") is not None:
        annotations.append(dict(x=b["put_wall"], y=0, yref="paper",
                                text=f"Put wall {b['put_wall']:.0f}", showarrow=False,
                                font=dict(color="#e74c3c", size=11), yshift=-12, xanchor="right"))

    fig.update_layout(
        title=f"GEX — {INDEX} · {b.get('label', key)} · expiry {b.get('expiry') or '—'} ({b.get('ttm_actual', '?')} BD) · {b.get('n_contracts', 0)} contracts",
        template="plotly_dark", barmode="relative", bargap=0.12,
        height=460, margin=dict(l=60, r=70, t=56, b=80),
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0),
        xaxis=dict(title="Strike", range=[x0, x1]),
        yaxis=dict(title="GEX ($B / 1%)", range=[-mag * 1.2, mag * 1.2], zeroline=True),
        yaxis2=dict(title="Agg GEX ($B)", overlaying="y", side="right",
                    range=[-cum_mag * 1.15, cum_mag * 1.15], showgrid=False),
        shapes=shapes, annotations=annotations,
    )
    fig.show()
    print("buckets:", ", ".join(f"{k}={v.get('n_contracts', 0)}" for k, v in buckets.items()))


plot_gex(data, "0")

buckets: 0=286, 1=304, 2=414, 3=542, 4=288, 5=285


In [5]:
from ssr import atmf_skew_slope, fetch_es_futures, rolling_ssr

# Realized SSR: R = (d σ_ATMF / d ln F) / S_T
# 1 sticky-strike, 0 sticky-delta. S_T = today's 25Δ/30d skew. σ = HMM IV (VIX). F = ES.
SSR_WINDOW = 10
SSR_DTE = 30

es, es_src = fetch_es_futures(lookback_days=90)
fut = es["close"]
iv = pd.Series(data.get("hmm_iv") or [], index=pd.to_datetime(data.get("hmm_dates") or []))
iv.index = iv.index.tz_localize(None).normalize()
slope = atmf_skew_slope(data.get("aiv"), data.get("psk"), dte=SSR_DTE)
ssr_df = rolling_ssr(fut, iv, slope, window=SSR_WINDOW)
if ssr_df.empty:
    raise RuntimeError("SSR empty — need overlapping ES futures and IV dates.")

latest = ssr_df.dropna(subset=["ssr"]).iloc[-1]
regime = "sticky-strike" if latest.ssr > 0.5 else "sticky-delta"
display(pd.DataFrame(
    [
        ("ES source", es_src),
        ("ES last", f"{float(fut.iloc[-1]):,.2f}  ({fut.index[-1].date()})"),
        ("S_T (25Δ slope)", f"{slope:.1f} vol pts / ln K"),
        (f"SSR {SSR_WINDOW}d", f"{latest.ssr:.2f}  ({regime})"),
        ("SSR 1d", f"{latest.ssr_1d:.2f}"),
    ],
    columns=["metric", "value"],
))

fig = go.Figure()
fig.add_scatter(x=ssr_df.index, y=ssr_df["ssr"], name=f"{SSR_WINDOW}d SSR", line=dict(color="#5dade2", width=2))
fig.add_hline(y=0, line=dict(color="#8f96a3", width=1, dash="dot"))
fig.add_hline(y=1, line=dict(color="#27ae60", width=1, dash="dash"))
fig.update_layout(
    title=f"Realized skew stickiness — {INDEX}<br><sup>1 sticky-strike · 0 sticky-delta · F=ES · σ={data.get('hmm_iv_label') or 'VIX'}</sup>",
    template="plotly_dark", height=360, hovermode="x unified",
    yaxis_title="SSR", xaxis=dict(type="date", rangeslider=dict(visible=False)),
    legend=dict(orientation="h", y=1.12),
    annotations=[
        dict(x=1, y=1, xref="paper", yref="y", text="sticky-strike", showarrow=False, font=dict(color="#27ae60", size=10), xanchor="right"),
        dict(x=1, y=0, xref="paper", yref="y", text="sticky-delta", showarrow=False, font=dict(color="#8f96a3", size=10), xanchor="right"),
    ],
)
fig.show()

,metric,value
0,ES source,yahoo ES=F
1,ES last,"7,700.25 (2026-08-26)"
2,S_T (25Δ slope),-122.9 vol pts / ln K
3,SSR 10d,0.90 (sticky-strike)
4,SSR 1d,0.08


In [6]:
from ssr import implied_ssr

# Implied SSR (Bergomi, today's smile only): R = 2 + d ln|S_T| / d ln T
# 1 sticky-strike, 2 short-dated local vol. No ES / VIX — skew vs DTE on the SVI grid.
# Realized in the cell above is on the same scale (1 sticky-strike, 0 sticky-delta).
IMP_DTE = SSR_DTE if "SSR_DTE" in dir() else 30

imp = implied_ssr(data["surface_x"], data["surface_y"], data.get("surface_sv") or data["surface_z"])
if imp.empty or imp["ssr"].dropna().empty:
    raise RuntimeError("Implied SSR empty — need ≥3 tenors on the live IV grid.")

r30 = float(np.interp(IMP_DTE, imp["dte"], imp["ssr"]))
s30 = float(np.interp(IMP_DTE, imp["dte"], imp["S"]))
gamma = float(imp["gamma"].iloc[0])
r_star = float(imp["ssr_power"].iloc[0])
r_real_bergomi = float(latest.ssr) if "latest" in dir() else float("nan")
regime = "sticky-strike" if r30 < 1.5 else "LV-like"

display(pd.DataFrame(
    [
        (f"Implied SSR {IMP_DTE:.0f}d", f"{r30:.2f}  ({regime})"),
        ("Implied SSR power-law", f"{r_star:.2f}  (γ={gamma:.2f})"),
        ("Realized 10d (Bergomi scale)", f"{r_real_bergomi:.2f}  (= realized cell)" if np.isfinite(r_real_bergomi) else "—"),
        (f"S_T {IMP_DTE:.0f}d (ATM dσ/d ln K)", f"{s30:.1f} vol pts / ln K"),
        ("Tenors", f"{int(imp.dte.min())}–{int(imp.dte.max())}d  n={len(imp)}"),
    ],
    columns=["metric", "value"],
))

fig = go.Figure()
fig.add_scatter(x=imp["dte"], y=imp["ssr"], name="implied SSR", mode="lines+markers",
                line=dict(color="#e67e22", width=2), marker=dict(size=8))
fig.add_scatter(x=imp["dte"], y=imp["ssr_power"], name=f"power-law {r_star:.2f}",
                line=dict(color="#e67e22", width=1, dash="dot"))
fig.add_hline(y=1, line=dict(color="#8f96a3", width=1, dash="dot"))
fig.add_hline(y=2, line=dict(color="#27ae60", width=1, dash="dash"))
fig.update_layout(
    title=f"Implied skew stickiness — {INDEX}<br><sup>Bergomi  1 sticky-strike · 2 short LV · SVI smile, no history</sup>",
    template="plotly_dark", height=360, hovermode="x unified",
    yaxis_title="SSR", xaxis_title="DTE",
    legend=dict(orientation="h", y=1.12),
    annotations=[
        dict(x=1, y=1, xref="paper", yref="y", text="sticky-strike", showarrow=False, font=dict(color="#8f96a3", size=10), xanchor="right"),
        dict(x=1, y=2, xref="paper", yref="y", text="short LV", showarrow=False, font=dict(color="#27ae60", size=10), xanchor="right"),
    ],
)
fig.show()


,metric,value
0,Implied SSR 30d,2.04 (LV-like)
1,Implied SSR power-law,2.09 (γ=-0.09)
2,Realized 10d (Bergomi scale),0.90 (= realized cell)
3,S_T 30d (ATM dσ/d ln K),-58.9 vol pts / ln K
4,Tenors,14–60d n=5


In [7]:
def high_vol_shapes(dates, regimes):
    shapes, start = [], None
    for i, (d, r) in enumerate(zip(dates, regimes)):
        high = r == "high_vol"
        if high and start is None:
            start = d
        elif not high and start is not None:
            shapes.append(dict(
                type="rect", xref="x", yref="paper", x0=start, x1=dates[i - 1], y0=0, y1=1,
                fillcolor="rgba(255,0,0,0.25)", line=dict(width=0), layer="below",
            ))
            start = None
    if start is not None:
        shapes.append(dict(
            type="rect", xref="x", yref="paper", x0=start, x1=dates[-1], y0=0, y1=1,
            fillcolor="rgba(255,0,0,0.25)", line=dict(width=0), layer="below",
        ))
    return shapes


def plot_hmm(data):
    dates = data.get("hmm_dates") or []
    if not dates:
        print("HMM history empty.")
        return
    shapes = high_vol_shapes(dates, data.get("hmm_regimes") or [])
    if data.get("hmm_today_date"):
        shapes.append(dict(
            type="line", xref="x", yref="paper",
            x0=data["hmm_today_date"], x1=data["hmm_today_date"], y0=0, y1=1,
            line=dict(color="#2ca02c", width=1.5, dash="dash"),
        ))
    fig = go.Figure()
    fig.add_candlestick(
        x=dates, open=data["hmm_opens"], high=data["hmm_highs"],
        low=data["hmm_lows"], close=data["hmm_prices"], name=f"{INDEX} up/down",
        increasing_line_color="#2ca02c", increasing_fillcolor="#2ca02c",
        decreasing_line_color="#d62728", decreasing_fillcolor="#d62728",
    )
    fig.add_scatter(x=dates, y=data["hmm_prices"], name="close",
                    line=dict(color="#d0d4dc", width=1.1))
    fig.update_layout(
        title=f"HMM Signal Window — {INDEX}<br><sup>P(low vol today) = {data['hmm_prob_today']}%  |  P(low vol tmr) = {data['hmm_prob_tmr']}%</sup>",
        template="plotly_dark", height=420, hovermode="x unified",
        legend=dict(orientation="h", y=1.08), shapes=shapes,
        yaxis_title=INDEX, xaxis=dict(type="date", rangeslider=dict(visible=False)),
    )
    fig.show()

    iv_label = data.get("hmm_iv_label") or "VIX"
    vol = go.Figure()
    vol.add_scatter(x=dates, y=data["hmm_rv"], name="22d RV %", line=dict(color="#1f77b4", width=1.2))
    vol.add_scatter(x=dates, y=data["hmm_iv"], name=f"{iv_label} IV", line=dict(color="#ff7f0e", width=1.2))
    if data.get("hmm_iv_22d_ago") is not None and np.isfinite(data["hmm_iv_22d_ago"]):
        vol.add_scatter(
            x=[dates[0], dates[-1]], y=[data["hmm_iv_22d_ago"], data["hmm_iv_22d_ago"]],
            name="IV 22d ago", line=dict(color="#888888", width=1.2, dash="dot"),
        )
    vol.update_layout(
        title="RV vs IV", template="plotly_dark", height=280, hovermode="x unified",
        legend=dict(orientation="h", y=1.15), shapes=high_vol_shapes(dates, data.get("hmm_regimes") or []),
        yaxis_title="Annualized vol %", xaxis=dict(type="date", rangeslider=dict(visible=False)),
    )
    vol.show()


plot_hmm(data)

In [8]:
metrics = data.get("structure_metrics") or []
if metrics:
    display(Markdown("**Structure metrics**"))
    display(pd.DataFrame(metrics))
else:
    print("No structure metrics.")

anoms = data.get("anomalies") or []
display(Markdown(f"**Surface anomalies ({len(anoms)})**"))
if anoms:
    display(pd.DataFrame(anoms))
else:
    print("None.")

**Structure metrics**

,key,metric,insight
0,sentiment,Market State: NEUTRAL,Vol premium and SSR imply balanced risk; no ex...
1,vol_level,Implied Vol (ATM 30d): 12.7% | VIX Base level:...,"Vol premium persists, but no spike."
2,skew,Skew Steepness Premium: 25d option slope is 2....,"Skew flat, no put/call imbalance."
3,butterfly,Wings Fly Curvature: 25d butterfly represents ...,"Wings muted, tail risk subdued."


**Surface anomalies (5)**

,kind,surface,ks,dte,value,baseline,score,detail
0,lv_explosion,lv,1.100,14.0,43.96,16.20,2.71,Local vol 44.0% vs IV 16.2% (ratio 2.7x)
1,lv_explosion,lv,1.100,10.0,51.17,19.01,2.69,Local vol 51.2% vs IV 19.0% (ratio 2.7x)
2,lv_explosion,lv,1.125,10.0,57.70,22.49,2.57,Local vol 57.7% vs IV 22.5% (ratio 2.6x)
3,lv_explosion,lv,0.875,45.0,59.13,23.65,2.50,Local vol 59.1% vs IV 23.7% (ratio 2.5x)
4,lv_explosion,lv,0.875,10.0,78.71,32.08,2.45,Local vol 78.7% vs IV 32.1% (ratio 2.5x)
